Обработка записанных логов при движения КС по угольному разрезу

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

DATA_DIR = ROOT_DIR / "data"
REC_DIR = DATA_DIR / "Records"
PROC_DIR = DATA_DIR / "Processed"

from src.preprocessing import preprocess as pp
from src.preprocessing import segmentation as sg
from src.preprocessing.real_data_adapter import prepare_real_segment
from src.preprocessing.run_real import run_segment, run_all_methods

from src.config.constants import INIT_ANGLE_IMU

REC_DIR.mkdir(exist_ok=True)
PROC_DIR.mkdir(exist_ok=True)

print("ROOT_DIR     :", ROOT_DIR)
print("RECORDS_DIR  :", REC_DIR,   "|", REC_DIR.exists())
print("PROCESSED_DIR:", PROC_DIR, "|", PROC_DIR.exists())

Выбор лога

In [ ]:
REC_SELECT = "2024.09.05_utro"

REC_SELECT_DIR  = REC_DIR / REC_SELECT  # папка с 4 CSV этого лога
PROC_SELECT_DIR = PROC_DIR / REC_SELECT # папка с parquet этого лога
SEG_ROOT = PROC_DIR / "segments" # куда лягут/откуда берутся сегменты

print("Лог      :", REC_SELECT)
print("REC_DIR  :", REC_SELECT_DIR,  "|", REC_SELECT_DIR.exists())
print("PROC_DIR :", PROC_SELECT_DIR, "|", PROC_SELECT_DIR.exists())

In [ ]:
# print(pd.read_csv(PATH_IMU, nrows=5).columns.tolist())
# print(pd.read_csv(PATH_GNSS0, nrows=5).columns.tolist())
# print(pd.read_csv(PATH_VEL, nrows=5).columns.tolist())

Выгружаем лог в parquet + различные преобразования (сетка, t0, поворот на константное смещение и т.д.)

In [ ]:
# Вывод функции pp.run (словарь):
#  result = {"gnss0": g0, "gnss1": g1, "vel0": v0, "vel1": v1,
#            "imu": imu, "meta": meta, "window": w}

out = pp.run(
    rec_dir = REC_SELECT_DIR,
    out_dir = PROC_SELECT_DIR,
    imu_tilt_deg = INIT_ANGLE_IMU,     # (Ox, Oy, Oz) в градусах
    warmup_sec = 30,
)
gnss0, imu = out["gnss0"], out["imu"]

Нарезка сценариев по логам

In [ ]:
data = sg.load_processed(PROC_SELECT_DIR)
ui = sg.FigureTabs(data) # save_dir=PROC_SELECT_DIR
ui.show()

segments = [
    sg.Seg(0,    580,  "stop_EngOff"),
    sg.Seg(600,  950,  "stop_EngOn"),
    sg.Seg(13140,  13500,  "dvij"),
    sg.Seg(13800,  14280,  "dvij"),
]
sg.validate_segments(segments, total_dur=ui.t_max)
ui.refresh_overview(segments)          # обзор с подсветкой размеченного — контроль
sg.cut_and_save(data, segments, PROC_SELECT_DIR / "segments")

Прогоняем реальный лог через фильтры

In [ ]:
SEG = PROC_SELECT_DIR / "segments/dvij/dvij_1"

# S = prepare_real_segment(SEG, lowpass_fc=15.0, align_sec=3.0)

Сравнение трёх методов на 1 сегменте

In [6]:
SEG = PROC_SELECT_DIR / "segments/dvij/dvij_1"

run_all_methods(str(SEG))

[EKF] отчёт: /home/rsadovec/prj/KAMAZ_Jupyter/data/Processed/2024.09.05_utro/segments/dvij/dvij_1/reports/ekf
[UKF] отчёт: /home/rsadovec/prj/KAMAZ_Jupyter/data/Processed/2024.09.05_utro/segments/dvij/dvij_1/reports/ukf
[FGO] отчёт: /home/rsadovec/prj/KAMAZ_Jupyter/data/Processed/2024.09.05_utro/segments/dvij/dvij_1/reports/fgo
[COMPARE] сравнительный отчёт: /home/rsadovec/prj/KAMAZ_Jupyter/data/Processed/2024.09.05_utro/segments/dvij/dvij_1/reports/compare
  RTK fix-доля: 100.0%


{'results': {'ekf': (array([[ 2.13966042e-08,  3.70045373e-08,  2.81240517e-10],
          [ 6.75808188e-06,  1.47319404e-05, -2.03559844e-07],
          [ 1.21892206e-05,  2.90142956e-05, -6.70744184e-07],
          ...,
          [-1.27669758e+02, -3.70803415e+02,  3.61297370e+01],
          [-1.27645648e+02, -3.70777229e+02,  3.61285935e+01],
          [-1.27572877e+02, -3.71111383e+02,  3.61728461e+01]]),
   array([[ 3.00551414e-03,  5.93752389e-03, -3.00738417e-05],
          [ 2.38383408e-03,  5.81842483e-03, -1.32999026e-04],
          [ 1.96107691e-03,  5.60745933e-03, -2.40748445e-04],
          ...,
          [ 9.61758298e+00,  1.04976318e+01, -4.57473006e-01],
          [ 9.67062922e+00,  1.04514402e+01, -4.57307347e-01],
          [ 9.72786391e+00,  1.02799056e+01, -4.68172994e-01]]),
   array([[-7.78675215e-04, -1.47099267e+00, -5.52440978e-01],
          [-1.17751703e-03, -1.47054342e+00, -5.52319491e-01],
          [-1.28410552e-03, -1.47064328e+00, -5.52097768e-01],
   

Один метод на 1 сегменте

In [ ]:
SEG = PROC_SELECT_DIR / "segments/dvij/dvij_1"

res = run_segment(str(SEG), method="fgo", show=True)